# Day 39: Multiple Linear Regression & Assumptions

## Fitting with Multiple Features
Simple linear regression predicts a target from just one input, but real-world outcomes almost always depend on several factors at once - a house's price depends on both its size and its number of bedrooms, not just one or the other. Multiple linear regression extends the same idea to several input features (X can have many columns), fitting one coefficient per feature that captures that feature's effect on the target while holding the others constant. Below, `X` contains two features (`size_sqft` and `bedrooms`) and `y` is the target (`price`); `LinearRegression().fit(X, y)` finds the best-fitting linear combination of those features. `model.coef_` prints one coefficient per feature in the same order as the columns of `X` (how much price changes per unit increase in that feature, holding the other fixed), and `model.intercept_` is the baseline predicted price when all features are zero.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# Small housing dataset: predict price from size and bedroom count
df = pd.DataFrame({
    'size_sqft': [1000, 1500, 1200, 1800, 2000, 900],
    'bedrooms': [2, 3, 2, 4, 4, 1],
    'price': [200000, 280000, 230000, 340000, 380000, 150000]
})
X = df[['size_sqft', 'bedrooms']]
y = df['price']
# Fit a linear model using both features simultaneously
model = LinearRegression().fit(X, y)
print('Coefficients:', model.coef_)
print('Intercept:', model.intercept_)

## Detailed Stats Summary
scikit-learn's `LinearRegression` is great for prediction but doesn't tell you whether a feature's effect is statistically meaningful or could just be noise - for that, data scientists often turn to the `statsmodels` library, which is built for statistical inference rather than pure prediction. `sm.add_constant(X)` explicitly adds a column of 1s to represent the intercept, since unlike scikit-learn, statsmodels' `OLS` (Ordinary Least Squares) doesn't add one automatically. Fitting `sm.OLS(y, X_const).fit()` produces the same underlying linear model as before, but `.summary()` prints a detailed report including each coefficient's p-value (is this feature's effect significantly different from zero?), its confidence interval, the R-squared (how much of the variance in price this model explains), and other diagnostics - this is the kind of output you'd include in a report to justify which features actually matter.

In [ ]:
# pip install statsmodels
import statsmodels.api as sm

# statsmodels requires manually adding a constant column for the intercept term
X_const = sm.add_constant(X)
ols_model = sm.OLS(y, X_const).fit()
# summary() gives coefficients, p-values, confidence intervals, R-squared, etc.
print(ols_model.summary())

## Assumption: Linearity
Linear regression assumes the relationship between features and target is genuinely linear - if it isn't, the model will systematically over- or under-predict in certain ranges, and checking the residuals (the errors: actual value minus predicted value) is the standard way to catch that. If the linearity assumption holds, residuals should be scattered randomly around zero with no visible pattern when plotted against the predicted values; if instead you see a curve, funnel shape, or other structure, that's a sign the model is missing something (like a non-linear relationship or a missing feature). The code below computes `residuals = y - predictions` and plots them against the predicted prices, with a red dashed horizontal line at zero as a visual reference - points scattered evenly above and below that line with no obvious shape support the linearity assumption; a visible pattern would suggest the model needs adjustment. Note that with only 6 data points here, there's too little data to draw a strong conclusion either way - this is meant to illustrate the technique.

In [ ]:
import matplotlib.pyplot as plt

# Residual = actual price minus the model's predicted price, for each row
predictions = model.predict(X)
residuals = y - predictions
plt.scatter(predictions, residuals)
# Reference line at 0: residuals should scatter randomly around this with no pattern
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted')
plt.ylabel('Residual')
plt.title('Residuals vs Predicted')
plt.show()

## Assumption: Multicollinearity
Multiple regression also assumes the input features aren't too strongly correlated with each other (multicollinearity) - if two features move almost in lockstep, the model can't reliably tell which one is actually driving the target, which makes the individual coefficients unstable and hard to interpret (a coefficient's sign or size can swing wildly with small changes in the data, even though overall predictions stay reasonable). A quick way to screen for this is to compute the correlation matrix between input features directly - values close to 1 or -1 indicate two features carry largely overlapping information. Here, `size_sqft` and `bedrooms` are checked with `X.corr()`; a high positive correlation between them (which is realistic, since bigger houses tend to have more bedrooms) would be a signal to interpret their individual coefficients cautiously, since the model may be splitting credit for "size effect" between the two correlated features rather than measuring each one cleanly.

In [ ]:
# Correlation between input features - values near 1 or -1 signal multicollinearity
print(X.corr())